# Aula 08 - Motor de Inferência e Sistema Especialista SCADA
## Célula de Manufatura Flexível (FMS)

Neste notebook implementamos a classe **`SistemaEspecialistaSCADA`** para processamento de regras SE... ENTÃO baseadas em Lógica de Predicados de Primeira Ordem (FOL), executando diagnósticos de falhas sensoriais, regras de triagem e interlocks de emergência.

In [1]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass
class EstadoPlanta:
    s_base: bool = False       # ZS-201
    s_topo: bool = False       # ZS-202
    cor_r: bool = False        # AS-201 Red
    cor_g: bool = False        # AS-201 Green
    cor_b: bool = False        # AS-201 Blue
    silo_vazio: bool = False   # LS-101
    peca_saida: bool = False   # ZS-101
    emergencia: bool = False   # HS-301 / S_emerg
    sobrecarga: bool = False   # S_sobrecarga
    timeout_pistao: bool = False
    contagem_caixas: Dict[int, int] = field(default_factory=lambda: {1: 0, 2: 0, 3: 0})

class SistemaEspecialistaSCADA:
    def __init__(self):
        self.alarmes_ativos: List[str] = []
        self.acoes_atuadores: Dict[str, bool] = {
            'XV-101': False,
            'XV-301': False,
            'XV-302': False,
            'XV-303': False,
            'M-101': True
        }
        self.mensagens_diagnostico: List[str] = []

    def inferir(self, st: EstadoPlanta) -> Dict[str, Any]:
        self.alarmes_ativos.clear()
        self.mensagens_diagnostico.clear()
        for k in self.acoes_atuadores:
            self.acoes_atuadores[k] = False
        self.acoes_atuadores['M-101'] = True

        # --- 1. REGRAS DE DIAGNÓSTICO (FOL) ---
        # R-DIAG-01: Inconsistência Geométrica
        if (not st.s_base) and st.s_topo:
            self.alarmes_ativos.append("HS-302 (a1)")
            self.mensagens_diagnostico.append("R-DIAG-01: Erro Geométrico (ZS-202 ativo s/ ZS-201) -> Peça Desalinhada")

        # R-DIAG-02: Ambiguidade de Cor
        cores_ativas = sum([st.cor_r, st.cor_g, st.cor_b])
        if cores_ativas > 1:
            self.alarmes_ativos.append("HS-302 (a1)")
            self.mensagens_diagnostico.append("R-DIAG-02: Ambiguidade Óptica de Cor (AS-201 com múltiplos canais)")

        # R-DIAG-03: Inconsistência de Silo
        if st.silo_vazio and st.peca_saida:
            self.alarmes_ativos.append("HS-302 (a1)")
            self.mensagens_diagnostico.append("R-DIAG-03: Conflito de Silo (LS-101 indica vazio e ZS-101 indica presença)")

        # R-DIAG-04: Timeout Mecânico
        if st.timeout_pistao:
            self.alarmes_ativos.append("HS-302 (a1)")
            self.mensagens_diagnostico.append("R-DIAG-04: Cilindro Travado (Timeout de avanço sem sinal de Fim de Curso)")

        # --- 2. REGRAS DE LOTEAMENTO E INTERVENÇÃO ---
        todas_cheias = all(qtd >= 10 for qtd in st.contagem_caixas.values())
        for cx, qtd in st.contagem_caixas.items():
            if qtd >= 10:
                self.mensagens_diagnostico.append(f"R-LOT-01: Caixa {cx} Cheia (10/10) - Intervenção Necessária")

        # --- 3. REGRAS DE SEGURANÇA E TRIP ---
        # R-SEC-01: Desarme Geral
        if st.emergencia or st.sobrecarga or todas_cheias:
            self.acoes_atuadores['M-101'] = False
            motivo = 'Emergência' if st.emergencia else ('Sobrecarga' if st.sobrecarga else 'Transbordo Geral')
            self.mensagens_diagnostico.append(f"R-SEC-01: Trip do Motor M-101 ativado por {motivo}")
            return self._gerar_saida()

        # Se houver falha de diagnóstico ativa, inibe atuação de triagem
        if self.alarmes_ativos:
            self.mensagens_diagnostico.append("Bloqueio de Triagem devido a Alarme Ativo")
            return self._gerar_saida()

        # --- 4. REGRAS DE TRIAGEM E ROTEAMENTO ---
        # R-TRI-01: Grande e Vermelha -> XV-301
        if st.s_base and st.s_topo and st.cor_r and (st.contagem_caixas[1] < 10):
            self.acoes_atuadores['XV-301'] = True
            self.mensagens_diagnostico.append("R-TRI-01: Atuando Braço Empurrador 1 (XV-301) -> Caixa 1")

        # R-TRI-02: Grande e Verde -> XV-302
        elif st.s_base and st.s_topo and st.cor_g and (st.contagem_caixas[2] < 10):
            self.acoes_atuadores['XV-302'] = True
            self.mensagens_diagnostico.append("R-TRI-02: Atuando Braço Empurrador 2 (XV-302) -> Caixa 2")

        # R-TRI-03: Pequena e Azul -> XV-303
        elif st.s_base and (not st.s_topo) and st.cor_b and (st.contagem_caixas[3] < 10):
            self.acoes_atuadores['XV-303'] = True
            self.mensagens_diagnostico.append("R-TRI-03: Atuando Braço Empurrador 3 (XV-303) -> Caixa 3")

        return self._gerar_saida()

    def _gerar_saida(self) -> Dict[str, Any]:
        return {
            'Alarmes': list(set(self.alarmes_ativos)),
            'Atuadores': dict(self.acoes_atuadores),
            'Diagnosticos': list(self.mensagens_diagnostico)
        }

print("[OK] SistemaEspecialistaSCADA carregado com sucesso!")

In [2]:
# TESTES AUTOMATIZADOS DO SISTEMA ESPECIALISTA
motor_expert = SistemaEspecialistaSCADA()

cenarios = [
    ("C1: Peça Grande Vermelha Normal", EstadoPlanta(s_base=True, s_topo=True, cor_r=True)),
    ("C2: Peça Pequena Azul Normal", EstadoPlanta(s_base=True, s_topo=False, cor_b=True)),
    ("C3: Falha Sensor Geometria (Topo s/ Base)", EstadoPlanta(s_base=False, s_topo=True, cor_r=True)),
    ("C4: Falha Sensor Cor (Vermelho + Verde)", EstadoPlanta(s_base=True, s_topo=True, cor_r=True, cor_g=True)),
    ("C5: Inconsistência no Silo", EstadoPlanta(silo_vazio=True, peca_saida=True)),
    ("C6: Emergência Ativada", EstadoPlanta(s_base=True, s_topo=True, cor_r=True, emergencia=True)),
    ("C7: Caixa 1 Cheia (10 peças)", EstadoPlanta(s_base=True, s_topo=True, cor_r=True, contagem_caixas={1: 10, 2: 3, 3: 5})),
    ("C8: Transbordo Geral (Todas Caixas Cheias)", EstadoPlanta(contagem_caixas={1: 10, 2: 10, 3: 10}))
]

relatorio = []
for nome, st in cenarios:
    res = motor_expert.inferir(st)
    relatorio.append({
        "Cenário": nome,
        "M-101": "LIGADO" if res['Atuadores']['M-101'] else "DESLIGADO",
        "Pistão Triagem": "XV-301" if res['Atuadores']['XV-301'] else ("XV-302" if res['Atuadores']['XV-302'] else ("XV-303" if res['Atuadores']['XV-303'] else "NENHUM")),
        "Alarme Disparado": ", ".join(res['Alarmes']) if res['Alarmes'] else "Nenhum",
        "Diagnóstico": res['Diagnosticos'][0] if res['Diagnosticos'] else "OK"
    })

print("=== RELATÓRIO DO MOTOR DE INFERÊNCIA DO SISTEMA ESPECIALISTA ===\n")
print(formatar_tabela(relatorio))
